In [ ]:
# STEP 1: Install Libraries
!pip install faiss-cpu transformers torch pandas

# STEP 2: Upload Dataset
from google.colab import files
uploaded = files.upload()

# STEP 3: Load Dataset
import pandas as pd

file_name = list(uploaded.keys())[0]
df = pd.read_csv(file_name)

print("Dataset Loaded Successfully")
print(df.head())

# STEP 4: Select Text Column
TEXT_COLUMN = df.columns[0]
item_texts = df[TEXT_COLUMN].astype(str).tolist()

print("Total items:", len(item_texts))
print("Sample item:", item_texts[0])

# STEP 5: Load Transformer Model
from transformers import AutoTokenizer, AutoModel
import torch
import numpy as np
import faiss

device = "cuda" if torch.cuda.is_available() else "cpu"

model_name = "sentence-transformers/all-MiniLM-L6-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device)
model.eval()

# STEP 6: Encode Texts into Embeddings
def encode_texts(texts, batch_size=32):
    all_embeddings = []

    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i+batch_size]

            tokens = tokenizer(
                batch,
                padding=True,
                truncation=True,
                return_tensors="pt"
            ).to(device)

            outputs = model(**tokens)

            embeddings = outputs.last_hidden_state.mean(dim=1)
            embeddings = embeddings.cpu().numpy().astype("float32")

            all_embeddings.append(embeddings)

    return np.vstack(all_embeddings)

item_embeddings = encode_texts(item_texts)

# STEP 7: Normalize Embeddings
faiss.normalize_L2(item_embeddings)

print("Embeddings shape:", item_embeddings.shape)

# STEP 8: Build FAISS Index
embedding_dim = item_embeddings.shape[1]
index = faiss.IndexFlatIP(embedding_dim)

index.add(item_embeddings)

print("FAISS index built with", index.ntotal, "items")

# STEP 9: Similarity Search Function
def get_similar_items(item_id, k=5):
    query_vector = item_embeddings[item_id:item_id+1]

    scores, indices = index.search(query_vector, k)

    return scores[0], indices[0]

# STEP 10: Test Similarity Retrieval
query_id = 0

scores, idxs = get_similar_items(query_id, k=5)

print("\nQUERY ITEM:")
print(item_texts[query_id])

print("\nSIMILAR ITEMS:")
for score, idx in zip(scores, idxs):
    print(f"-> {item_texts[idx]} (score={score:.3f})")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 38.9 MB/s eta 0:00:00


Saving amazon_categories.csv to amazon_categories.csv
Dataset Loaded Successfully
   id                     category_name
0   1          Beading & Jewelry Making
1   2                 Fabric Decorating
2   3       Knitting & Crochet Supplies
3   4              Printmaking Supplies
4   5  Scrapbooking & Stamping Supplies
Total items: 248
Sample item: 1


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embeddings shape: (248, 384)
FAISS index built with 248 items

QUERY ITEM:
1

SIMILAR ITEMS:
-> 1 (score=1.000)
-> 2 (score=0.805)
-> 3 (score=0.791)
-> 4 (score=0.769)
-> 7 (score=0.658)
